# Tool Conversion

The `convert.py` module converts Python functions and LangChain `Runnable` objects into tools.

It supports direct function conversion, decorator-based tool creation, automatic input-schema inference, explicit schemas, synchronous and asynchronous functions, structured tool responses, and provider-specific tool configuration.

### Functions

1. `tool`: Converts a Python callable or `Runnable` into a LangChain `BaseTool`.

   It may be used directly or as a decorator with or without arguments. Function signatures are used to infer structured input schemas unless schema inference is disabled.

   When converting a `Runnable`, a string tool name must be supplied and the Runnable must expose an object-shaped input schema.

   Tool-description precedence is:

   - The explicitly supplied `description`
   - The callable's docstring
   - The description from `args_schema`

   When `response_format` is `"content_and_artifact"`, the callable must return a two-item tuple containing message content and an artifact.

   * **Supported Forms:**
     ```python
     tool(
         callable
     ) -> BaseTool
     ```

     ```python
     tool(
         "tool_name",
         runnable,
         ...
     ) -> BaseTool
     ```

     ```python
     tool(
         "tool_name",
         ...
     ) -> Callable[
         [Callable[..., Any] | Runnable[Any, Any]],
         BaseTool
     ]
     ```

     ```python
     tool(
         ...
     ) -> Callable[
         [Callable[..., Any] | Runnable[Any, Any]],
         BaseTool
     ]
     ```

   * **Syntax:**
     ```python
     tool(
         name_or_callable: str | Callable[..., Any] | None = None, # Tool name or callable to convert
         runnable: Runnable[Any, Any] | None = None, # Runnable to convert
         *args: Any, # Extra positional arguments, which must be empty
         description: str | None = None, # Optional tool description
         return_direct: bool = False, # Whether to stop the agent loop after execution
         args_schema: ArgsSchema | None = None, # Explicit tool-input schema
         infer_schema: bool = True, # Whether to infer a schema from the callable
         response_format: Literal[
             "content",
             "content_and_artifact"
         ] = "content", # Interpretation of the tool's return value
         parse_docstring: bool = False, # Whether to parse Google-style argument descriptions
         error_on_invalid_docstring: bool = True, # Whether invalid docstrings raise an error
         extras: dict[str, Any] | None = None # Provider-specific tool configuration
     ) -> BaseTool | Callable[
         [Callable[..., Any] | Runnable[Any, Any]],
         BaseTool
     ]
     ```

2. `convert_runnable_to_tool`: Converts a `Runnable` into a `BaseTool`.

   A string-input Runnable becomes a simple `Tool`. A Runnable with structured input becomes a `StructuredTool`.

   When `args_schema` is provided, it is bound to the Runnable as its input type. Otherwise, the function uses the Runnable's object schema when available or creates a Pydantic schema from `arg_types` or the Runnable's annotated input type.

   The Runnable's name is used when `name` is omitted. A placeholder description based on the input JSON schema is generated when `description` is omitted.

   * **Syntax:**
     ```python
     convert_runnable_to_tool(
         runnable: Runnable[Any, Any], # Runnable to convert
         args_schema: TypeBaseModel | None = None, # Pydantic input schema
         *,
         name: str | None = None, # Optional tool name
         description: str | None = None, # Optional tool description
         arg_types: dict[str, type] | None = None # Input argument names and types
     ) -> BaseTool
     ```